# Vector Databases - Hands-On Tour

A single notebook walking through `../docs/`: create a collection &rarr;
insert/upsert &rarr; query &rarr; filter &rarr; stable IDs &rarr; lifecycle &rarr;
multi-tenancy &rarr; hybrid search &rarr; reranking. Everything runs against a
**local, persistent ChromaDB collection backed by a local Ollama
embedding model** - no API key, no signup, nothing leaves your machine.

## Setup

```bash
ollama pull nomic-embed-text
ollama pull llama3.1:8b       # only needed for the reranking section
pip install chromadb requests openai
```

A couple of behaviors in this notebook were verified directly against
a running ChromaDB instance rather than assumed from documentation -
worth doing yourself whenever a library's edge-case behavior actually
matters to your code.

In [ ]:
import chromadb
import requests
from chromadb import Documents, EmbeddingFunction, Embeddings

class OllamaEmbeddingFunction(EmbeddingFunction):
    def __init__(self, model="nomic-embed-text", url="http://localhost:11434/api/embeddings"):
        self.model = model
        self.url = url

    def __call__(self, input: Documents) -> Embeddings:
        return [
            requests.post(self.url, json={"model": self.model, "prompt": text}).json()["embedding"]
            for text in input
        ]

client = chromadb.PersistentClient(path="./chroma_notebook_demo")

## 1. Create a collection and query it

See `docs/01-What-Is-a-Vector-Database.md` and
`docs/05-Running-ChromaDB-Locally.md`.

In [ ]:
collection = client.get_or_create_collection(
    name="runbooks",
    embedding_function=OllamaEmbeddingFunction(),
    metadata={"hnsw:space": "cosine"},
)

collection.upsert(
    ids=["r1", "r2", "r3"],
    documents=[
        "To debug CrashLoopBackOff, check kubectl describe pod and logs --previous.",
        "When disk usage hits 90% on a primary DB, it's HIGH severity.",
        "To roll back a deployment, use kubectl rollout undo.",
    ],
    metadatas=[{"source": "runbook-01.md"}, {"source": "runbook-02.md"}, {"source": "runbook-03.md"}],
)

results = collection.query(query_texts=["one of our pods keeps restarting"], n_results=2)
for doc, dist in zip(results["documents"][0], results["distances"][0]):
    print(f"[distance={dist:.3f}] {doc}")

## 2. `add()` vs `upsert()` - verified, not assumed

See `docs/06-Inserting-and-Updating-Vectors.md`. This was tested
directly against a real ChromaDB instance: `add()` on a duplicate ID
does **not** raise - it silently does nothing. That's a more dangerous
failure mode than an error, because a broken re-run pipeline can look
successful while quietly failing to apply updates.

In [ ]:
collection.add(ids=["x1"], documents=["original version"])
collection.add(ids=["x1"], documents=["trying to add again"])  # no error!
print("after duplicate add():", collection.get(ids=["x1"])["documents"])  # still original

collection.upsert(ids=["x1"], documents=["updated via upsert"])
print("after upsert():", collection.get(ids=["x1"])["documents"])  # actually changed

## 3. Metadata filtering, applied during the search

See `docs/08-Metadata-Filtering.md`.

In [ ]:
collection.upsert(
    ids=["p1", "p2"],
    documents=["Vacation carry-over must be requested by November 1.", "Sick leave carry-over must be requested by December 15."],
    metadatas=[{"document_type": "vacation_policy"}, {"document_type": "sick_leave_policy"}],
)

filtered = collection.query(
    query_texts=["carry-over deadline"], n_results=5, where={"document_type": "vacation_policy"},
)
print([m["document_type"] for m in filtered["metadatas"][0]])

## 4. Stable IDs make re-indexing idempotent

See `docs/09-Stable-IDs-and-Idempotent-Upserts.md`.

In [ ]:
import hashlib

def stable_id(source, chunk_index):
    return hashlib.sha256(f"{source}|{chunk_index}".encode()).hexdigest()[:16]

def index_documents(documents):
    ids = [stable_id(d["source"], d["chunk_index"]) for d in documents]
    collection.upsert(
        ids=ids,
        documents=[d["text"] for d in documents],
        metadatas=[{"source": d["source"]} for d in documents],
    )

docs = [{"source": "runbook.md", "chunk_index": 0, "text": "Restart the pod by deleting it."}]
index_documents(docs)
print("after 1st run:", collection.count())
index_documents(docs)  # same input again
print("after 2nd run (identical):", collection.count())

## 5. Lifecycle: soft delete vs. hard delete

See `docs/11-Document-Lifecycle-Update-and-Delete.md`.

In [ ]:
collection.upsert(
    ids=["old1"], documents=["outdated policy content"], metadatas=[{"source": "old.md", "active": True}],
)

# soft delete - deactivate, don't remove
collection.update(ids=["old1"], metadatas=[{"active": False}])
active_results = collection.query(query_texts=["policy content"], n_results=5, where={"active": True})
print("active-only results exclude old1:", "old1" not in active_results["ids"][0])

# hard delete by metadata filter - removes every chunk from a source
collection.delete(where={"source": "old.md"})
print("count after hard delete:", collection.count())

## 6. Multi-tenant isolation, centralized in one function

See `docs/12-Multi-Tenancy-and-Access-Control.md`.

In [ ]:
def tenant_scoped_query(tenant_id, query_text, n_results=5):
    return collection.query(query_texts=[query_text], n_results=n_results, where={"tenant_id": tenant_id})

collection.upsert(
    ids=["t1-doc1", "t2-doc1"],
    documents=["Tenant A's internal runbook.", "Tenant B's internal runbook."],
    metadatas=[{"tenant_id": "tenant-a"}, {"tenant_id": "tenant-b"}],
)

result_a = tenant_scoped_query("tenant-a", "runbook")
print("tenant A only sees:", result_a["metadatas"][0])

## 7. Hybrid search: exact-term filter + vector ranking

See `docs/14-Hybrid-Search-in-a-Vector-Database.md`.

In [ ]:
collection.upsert(
    ids=["d1", "d2"],
    documents=[
        "Common causes of database connection issues include pool exhaustion.",
        "Error E4021 occurs when the connection pool hits its configured max size.",
    ],
)

exact = collection.query(query_texts=["what causes error E4021"], n_results=2, where_document={"$contains": "E4021"})
print(exact["documents"][0])

## Wrap-up

| Section | Concept | Docs chapter |
| --- | --- | --- |
| 1 | Collections and querying | 01, 05 |
| 2 | add() vs upsert() (verified behavior) | 06 |
| 3 | Metadata filtering | 08 |
| 4 | Stable IDs | 09 |
| 5 | Document lifecycle | 11 |
| 6 | Multi-tenancy | 12 |
| 7 | Hybrid search | 14 |

Not covered here but worth running from `../examples/`:
`02_scaling_and_latency.py` (watching latency vs. size as data grows)
and `10_reranking.py` (a second LLM-based scoring pass over a small
candidate set).

Next: `05-RAG` - this notebook's collection-building pattern is exactly
what that module's real pipeline does with a real PDF.